# Monthly UTrack precipitation- and evaporationshed maps

This notebook calculates and maps the twelve monthly UTrack precipitationsheds and evaporationsheds for 37.78228045733923° N, -100.90621369000884° E. It runs both UTrack calculators, then produces one map for each monthly result with a shared logarithmic colour scale within each shed type.

Edit the UTrack and ERA5 paths in the next cell before running the notebook.

In [5]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import xarray as xr

## Configuration

Set `utrack_directory` and `era5_directory` to the directories containing the monthly NetCDF input files. The defaults below are placeholders and must be replaced.

In [6]:
# Set paths to your data directories.
home = Path.home()
utrack_directory = home / "OneDrive - University of Kansas" / "Research" / "MoistureRecycling" / "UTrack"
era5_directory = home / "OneDrive - University of Kansas" / "Research" / "MoistureRecycling" / "RECON"

# The coordinate whose monthly precipitation- and evaporationsheds are calculated.
target_latitude = 37.78228045733923
target_longitude = -100.90621369000884

# UTrack's distribution used by this project stores moisture_flow as -10 times
# the natural log of the flow fraction. Select "fraction" only for already-decoded inputs.
flow_encoding = "encoded-log"

# Resolve the project root whether Jupyter was started in the repository or code directory.
current_directory = Path.cwd().resolve()
if (current_directory / "data").is_dir() and (current_directory / "figures+tables").is_dir():
    repository_root = current_directory
elif (current_directory.parent / "data").is_dir() and (current_directory.parent / "figures+tables").is_dir():
    repository_root = current_directory.parent
else:
    raise FileNotFoundError("Start Jupyter from the repository root or its code directory.")
code_directory = repository_root / "code" if (repository_root / "code").is_dir() else repository_root
precipitationshed_calculator_script = code_directory / "UTrack-Point_01a_MonthlyPrecipitationshed.py"
evaporationshed_calculator_script = code_directory / "UTrack-Point_01b_MonthlyEvaporationshed.py"
precipitationshed_output_directory = repository_root / "data" / "UTrack_monthly_precipitationshed_37.7823_-100.9062"
evaporationshed_output_directory = repository_root / "data" / "UTrack_monthly_evaporationshed_37.7823_-100.9062"
precipitationshed_map_output_directory = repository_root / "figures+tables" / "UTrack_monthly_precipitationshed_maps_37.7823_-100.9062"
evaporationshed_map_output_directory = repository_root / "figures+tables" / "UTrack_monthly_evaporationshed_maps_37.7823_-100.9062"

for directory, label in ((utrack_directory, "UTrack"), (era5_directory, "ERA5")):
    if not directory.is_dir():
        raise FileNotFoundError(f"{label} directory not found: {directory}")
for calculator_script in (precipitationshed_calculator_script, evaporationshed_calculator_script):
    if not calculator_script.is_file():
        raise FileNotFoundError(f"Monthly calculator not found: {calculator_script}")

precipitationshed_output_directory.mkdir(parents=True, exist_ok=True)
evaporationshed_output_directory.mkdir(parents=True, exist_ok=True)
precipitationshed_map_output_directory.mkdir(parents=True, exist_ok=True)
evaporationshed_map_output_directory.mkdir(parents=True, exist_ok=True)

## Calculate monthly precipitation- and evaporationsheds

This runs both calculators and writes twelve monthly NetCDF files for each shed type.

In [7]:
def run_calculator(calculator_script, output_directory):
    command = [
        sys.executable,
        str(calculator_script),
        "--latitude",
        str(target_latitude),
        "--longitude",
        str(target_longitude),
        "--utrack-directory",
        str(utrack_directory),
        "--era5-directory",
        str(era5_directory),
        "--output-directory",
        str(output_directory),
        "--flow-encoding",
        flow_encoding,
    ]
    subprocess.run(command, check=True)

run_calculator(precipitationshed_calculator_script, precipitationshed_output_directory)
run_calculator(evaporationshed_calculator_script, evaporationshed_output_directory)

## Write monthly maps

The precipitationshed maps show upwind evaporation contributions to precipitation at the selected target. The evaporationshed maps show downwind precipitation contributions from evaporation at the selected source. Each shed type uses a shared scale across its twelve months.

In [8]:
def load_monthly_contributions(output_directory, filename_prefix, contribution_variable):
    monthly_contributions = []
    for month in range(1, 13):
        input_path = output_directory / f"{filename_prefix}_month_{month:02d}.nc"
        if not input_path.is_file():
            raise FileNotFoundError(f"Expected monthly output not found: {input_path}")
        with xr.open_dataset(input_path) as dataset:
            if contribution_variable not in dataset:
                raise KeyError(f"{input_path} does not contain {contribution_variable}.")
            monthly_contributions.append((month, dataset[contribution_variable].load(), dict(dataset.attrs)))
    return monthly_contributions


def write_monthly_maps(shed_name, monthly_contributions, latitude_name, longitude_name, marker_latitude, marker_longitude, colorbar_label, output_directory):
    positive_values = [values.values[np.isfinite(values.values) & (values.values > 0)] for _, values, _ in monthly_contributions]
    positive_values = [values for values in positive_values if values.size]
    if not positive_values:
        raise ValueError(f"No positive {shed_name} contributions are available to map.")
    limits = (float(np.concatenate(positive_values).min()), float(np.concatenate(positive_values).max()))

    for month, contribution, attributes in monthly_contributions:
        longitudes = (contribution[longitude_name].values + 180) % 360 - 180
        longitude_order = np.argsort(longitudes)
        values = contribution.isel({longitude_name: longitude_order}).values
        values = np.ma.masked_where(~np.isfinite(values) | (values <= 0), values)
        figure, axis = plt.subplots(figsize=(12, 6), constrained_layout=True)
        image = axis.pcolormesh(longitudes[longitude_order], contribution[latitude_name].values, values, cmap="viridis", norm=LogNorm(*limits), shading="auto")
        axis.plot(float(attributes[marker_longitude]), float(attributes[marker_latitude]), marker="*", markersize=12, markeredgecolor="black", markerfacecolor="white", linestyle="none", label="Selected location")
        axis.set(title=f"UTrack monthly {shed_name}: month {month:02d}", xlabel="Longitude", ylabel="Latitude", xlim=(-180, 180), ylim=(-90, 90), aspect="equal")
        axis.legend(loc="upper right")
        figure.colorbar(image, ax=axis, label=colorbar_label)
        output_path = output_directory / f"utrack_{shed_name}_month_{month:02d}.png"
        figure.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.close(figure)
        print(f"Month {month:02d}: wrote {output_path}")


precipitationshed_contributions = load_monthly_contributions(precipitationshed_output_directory, "utrack_precipitationshed", "evaporation_contribution_m3")
write_monthly_maps("precipitationshed", precipitationshed_contributions, "sourcelat", "sourcelon", "selected_target_latitude_degrees_north", "selected_target_longitude_degrees_east", "Evaporation contribution (m3 month-1)", precipitationshed_map_output_directory)

evaporationshed_contributions = load_monthly_contributions(evaporationshed_output_directory, "utrack_evaporationshed", "precipitation_contribution_m3")
write_monthly_maps("evaporationshed", evaporationshed_contributions, "targetlat", "targetlon", "selected_source_latitude_degrees_north", "selected_source_longitude_degrees_east", "Precipitation contribution (m3 month-1)", evaporationshed_map_output_directory)

Month 01: wrote C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\figures+tables\UTrack_monthly_precipitationshed_maps_37.7823_-100.9062\utrack_precipitationshed_month_01.png
Month 02: wrote C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\figures+tables\UTrack_monthly_precipitationshed_maps_37.7823_-100.9062\utrack_precipitationshed_month_02.png
Month 03: wrote C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\figures+tables\UTrack_monthly_precipitationshed_maps_37.7823_-100.9062\utrack_precipitationshed_month_03.png
Month 04: wrote C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\figures+tables\UTrack_monthly_precipitationshed_maps_37.7823_-100.9062\utrack_precipitationshed_month_04.png
Month 05: wrote C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\figures+tables\UTrack_monthly_precipitationshed_maps_37.7823_-100.9062\utrack_precipitationshed_month_05.png
Month 06: wrote C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\figures+tables\UTrack_monthly_precipitationshed_maps_37.782